# STEP 2 — Colab 임베딩

**순서**
1. 런타임 → 런타임 유형 변경 → **T4 GPU** 선택
2. `chunks.json` 업로드
3. 셀 순서대로 실행
4. `vectors.npz` 다운로드 → PyCharm 프로젝트의 `data/export/`에 넣기

In [33]:
# ── 셀 1: GPU 확인 ──────────────────────────────────────────
import torch
print('CUDA 사용 가능:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

CUDA 사용 가능: True
GPU: Tesla T4


In [34]:
# ── 셀 2: 패키지 설치 ────────────────────────────────────────
!pip install -q sentence-transformers

In [35]:
# Colab에서 기존 파일 삭제 후 업로드
import os
if os.path.exists('chunks.json'):
    os.remove('chunks.json')
    print('삭제 완료')

from google.colab import files
uploaded = files.upload()  # 새 chunks.json 선택

삭제 완료


Saving chunks.json to chunks.json


In [36]:
import json
import random
import re

with open("chunks.json", encoding="utf-8") as f:
    chunks = json.load(f)

eval_data = []
qid = 1

for chunk in chunks:

    chunk_id = chunk["chunk_id"]
    text = chunk["text"]

    if len(text) < 80:
        continue

    article_match = re.search(r"제(\d+)조", text)

    templates = [
        "다음 내용은 어떤 규정을 설명하나요?",
        "이 조항은 무엇에 관한 내용인가요?",
        "다음 규정과 관련된 조문을 찾아주세요.",
        "계약 실무에서 아래 내용은 어떤 법적 근거에 해당하나요?"
    ]

    query = random.choice(templates) + "\n\n" + text[:120]

    eval_data.append({
        "query_id": f"eval_{qid:03d}",
        "query": query,
        "relevant_docs": [chunk_id]
    })

    qid += 1

random.shuffle(eval_data)

eval_data = eval_data[:200]

with open(
    "eval_dataset_v2.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        eval_data,
        f,
        ensure_ascii=False,
        indent=2
    )

print(len(eval_data))

200


In [29]:
# ── 셀 4: 청크 로드 ──────────────────────────────────────────
import json

with open('chunks.json', encoding='utf-8') as f:
    chunks = json.load(f)

# chunk_id 기준 중복 제거
seen = {}
for c in chunks:
    seen[c["chunk_id"]] = c
chunks = list(seen.values())

texts = [c['text'] for c in chunks]
chunk_ids = [c['chunk_id'] for c in chunks]

print(f'청크 수 (중복 제거 후): {len(chunks)}')
print(f'예시: {texts[0][:80]}...')

청크 수 (중복 제거 후): 2300
예시: 제1조(시행일) 이 법은 공포 후 3개월이 경과한 날부터 시행한다....


In [37]:
import gc
import torch
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

texts = [c["text"] for c in chunks]
chunk_ids = [c["chunk_id"] for c in chunks]

def evaluate_model(model_name, batch_size=8):

    print(f"\n===== {model_name} =====")

    model = SentenceTransformer(
        model_name,
        device="cuda"
    )

    # 문서 임베딩
    doc_vectors = model.encode(
        texts,
        batch_size=batch_size,
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=True
    )

    recall1 = 0
    recall5 = 0
    recall10 = 0
    mrr = 0

    for item in eval_data:

        query = item["query"]
        gt_docs = set(item["relevant_docs"])

        qvec = model.encode(
            query,
            normalize_embeddings=True,
            convert_to_numpy=True
        )

        scores = cosine_similarity(
            [qvec],
            doc_vectors
        )[0]

        ranking = np.argsort(scores)[::-1]

        ranked_docs = [
            chunk_ids[idx]
            for idx in ranking
        ]

        if any(doc in gt_docs for doc in ranked_docs[:1]):
            recall1 += 1

        if any(doc in gt_docs for doc in ranked_docs[:5]):
            recall5 += 1

        if any(doc in gt_docs for doc in ranked_docs[:10]):
            recall10 += 1

        for rank, doc in enumerate(ranked_docs, start=1):
            if doc in gt_docs:
                mrr += 1 / rank
                break

    n = len(eval_data)

    result = {
        "Model": model_name,
        "Recall@1": round(recall1 / n, 4),
        "Recall@5": round(recall5 / n, 4),
        "Recall@10": round(recall10 / n, 4),
        "MRR": round(mrr / n, 4)
    }

    # GPU 메모리 해제
    del doc_vectors
    del model

    gc.collect()
    torch.cuda.empty_cache()

    return result

In [17]:
import pandas as pd

result = evaluate_model(
    "BAAI/bge-m3",
    batch_size=8
)

df = pd.DataFrame([result])

display(df)


===== BAAI/bge-m3 =====


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Batches:   0%|          | 0/315 [00:00<?, ?it/s]

,Model,Recall@1,Recall@5,Recall@10,MRR
0,BAAI/bge-m3,0.62,0.905,0.955,0.7329


In [18]:
result = evaluate_model(
    "nlpai-lab/KURE-v1",
    batch_size=8
)

df = pd.DataFrame([result])

display(df)


===== nlpai-lab/KURE-v1 =====


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Batches:   0%|          | 0/315 [00:00<?, ?it/s]

,Model,Recall@1,Recall@5,Recall@10,MRR
0,nlpai-lab/KURE-v1,0.655,0.89,0.96,0.7564


In [19]:
# result = evaluate_model(
#     "Qwen/Qwen3-Embedding-0.6B",
#     batch_size=1
# )

# df = pd.DataFrame([result])

# display(df)

In [38]:
import numpy as np
import torch
from sentence_transformers import SentenceTransformer

# 1. 모델 로드 (메모리 절약을 위해 float16 사용)
model_name = "nlpai-lab/KURE-v1"
model = SentenceTransformer(
    model_name,
    device="cuda",
    model_kwargs={"torch_dtype": torch.float16}
)

# 2. 전전 셀에서 로드한 texts 배치를 임베딩
print("전체 청크 임베딩 시작...")
doc_vectors = model.encode(
    texts,
    batch_size=8,
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=True
)

# 3. 데이터와 ID를 함께 npz 파일로 저장
# chunk_ids도 함께 저장해야 나중에 검색했을 때 어떤 본문인지 매칭할 수 있습니다.
np.savez(
    "vectors.npz",
    vectors=doc_vectors,
    chunk_ids=np.array(chunk_ids)
)

print("vectors.npz 저장 완료! 왼쪽 파일 탭에서 다운로드하세요.")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

전체 청크 임베딩 시작...


Batches:   0%|          | 0/315 [00:00<?, ?it/s]

vectors.npz 저장 완료! 왼쪽 파일 탭에서 다운로드하세요.


In [39]:
import json
with open('chunks.json', encoding='utf-8') as f:
    chunks = json.load(f)
print('총 개수:', len(chunks))
print('샘플:', [c['chunk_id'] for c in chunks[:5]])

총 개수: 2513
샘플: ['LCA_1', 'LCA_2', 'LCA_3', 'LCA_4', 'LCA_5_1_5']
